In [ ]:
import micropip
await micropip.install("matplotlib")
await micropip.install("scipy")
await micropip.install("ipywidgets")

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

%matplotlib inline

def correlation(f, g, ih):
    f = f.astype(np.complex128)
    g = g.astype(np.complex128)
    g_shifted = np.roll(g, ih)
    corr = np.sum(np.conjugate(g_shifted) * f)
    return corr

def full_correlation(f, g):
    N = f.shape[0]
    full_corr = np.zeros(N, dtype=np.complex128)
    for ih in range(-N // 2, N // 2):
        full_corr[ih + N // 2] = correlation(f, g, ih)
    return full_corr


def representacion(ih_time):
    """ih_time: shift in the same units as the global time array t.
       Uses global dt to convert to integer index ih."""
    ih = int(round(ih_time / dt))
    N = f.shape[0]
    g_shifted = np.roll(g, ih)
    integrand = np.conj(g_shifted) * f

    fig, ax = plt.subplots(2, 2, figsize=(7, 5))

    # Top-left: f
    ax[0, 0].plot(t, np.real(f), label='Re(f)')
    ax[0, 0].plot(t, np.abs(f), '--', label='|f|')
    ax[0, 0].set_title('f(t)')
    ax[0, 0].legend(fontsize=8)
    #ax[0, 0].set_xlim(0, N)

    # Top-right: g and shifted g
    ax[0, 1].plot(t, np.real(g), label='Re(g)', alpha=0.4)
    ax[0, 1].plot(t, np.real(g_shifted), label=f'Re(g shifted, h={ih_time:.2f})')
    ax[0, 1].plot(t, np.abs(g_shifted), '--', label='|g shifted|')
    ax[0, 1].set_title(f'g(t) shifted by h = {ih_time:.2f}')
    ax[0, 1].legend(fontsize=8)

    # Bottom-left: integrand
    ax[1, 0].plot(t, np.abs(g_shifted), alpha=0.4, label='|g shifted|')
    ax[1, 0].plot(t, np.abs(f), alpha=0.4, label='|f|')
    ax[1, 0].plot(t, np.real(integrand), label='Re(g* · f)', linewidth=2)
    #ax[1, 0].plot(t, np.abs(integrand), '--', label='|g* · f|', linewidth=2)
    ax[1, 0].fill_between(t, np.real(integrand), alpha=0.2)
    ax[1, 0].set_title(f'Integrand  →  C(h={ih_time:.2f}) = {correlation(f, g, ih):.3f}')
    ax[1, 0].legend(fontsize=8)

    # Bottom-right: full correlation — x-axis in time units
    ih_axis_time = ih_axis * dt
    ax[1, 1].plot(ih_axis_time, np.abs(full_corr), label='|C(h)|')
    #ax[1, 1].plot(ih_axis_time, np.real(full_corr), label='Re(C(h))', alpha=0.6)
    ax[1, 1].axvline(x=ih_time, color='r', linestyle='--', label=f'h={ih_time:.2f}')
    ax[1, 1].scatter([ih_time], [np.abs(full_corr[ih + N // 2])], c='r', zorder=5)
    ax[1, 1].set_title('Correlation C(h)')
    ax[1, 1].legend(fontsize=8)
    ax[1, 1].set_xlabel('h (time units)')

    plt.tight_layout()
    plt.show()

# El producto de correlación

Elm producto de correlación es una medida de la regularidad relativa entre dos funciones. Es por eso que también se le considera una forma de avaluar la coherencia entre dos funciones.

Nosotros lo aplicaremos a campos electromagnéticos en el tiempo.

Tomamos un eje temporal que se extiende durante 100 $fs$ (un $fs$ es $10^{-15}$ s). Típicamente 

# Correlación de dos envolventes

Comencemos evaluando la correlación entre dos funciones arbitrarias, que podemos considerar envolventes de dos ondas:

$f= e^{-(t-t_0)^2/10}$

$g= e^{-(t-t_0)^2/30}$



In [18]:
npts = 500
t = np.linspace(-50, 50, npts)
dt = t[1] - t[0]               # time step (fs) — used by representacion
ix = np.arange(npts)
f = np.exp(-t**2 / 50)
g = np.exp(-t**2 / 70)



full_corr = full_correlation(f, g)
ih_axis = np.arange(-npts // 2, npts // 2)

slider = widgets.FloatSlider(
    value=0.0, min=t[0], max=t[-1], step=4*dt,
    description='h (fs):', continuous_update=True,
    layout=widgets.Layout(width='450px')
)

btn_left  = widgets.Button(description='◀', layout=widgets.Layout(width='40px'))
btn_right = widgets.Button(description='▶', layout=widgets.Layout(width='40px'))

def step_left(b):
    slider.value = max(slider.min, slider.value - slider.step)

def step_right(b):
    slider.value = min(slider.max, slider.value + slider.step)

btn_left.on_click(step_left)
btn_right.on_click(step_right)

out = widgets.interactive_output(representacion, {'ih_time': slider})
controls = widgets.HBox([btn_left, slider, btn_right])
display(widgets.VBox([controls, out]))

#widgets.interact(representacion, ih_time=slider)

plt.close()

In [ ]:
npts = 500
t = np.linspace(-50, 50, npts)
dt = t[1] - t[0]               # time step (fs) — used by representacion
ix = np.arange(npts)
wf=2*np.pi/2
f = np.exp(-t**2 / 50)*np.exp(1j*wf*t)
wg=2*np.pi/2/2
g = np.exp(-t**2 / 70)*np.exp(1j*wg*t)



full_corr = full_correlation(f, g)
ih_axis = np.arange(-npts // 2, npts // 2)

slider = widgets.FloatSlider(
    value=0.0, min=t[0], max=t[-1], step=dt,
    description='h (fs):', continuous_update=True,
    layout=widgets.Layout(width='500px')
)

widgets.interact(representacion, ih_time=slider)
plt.close()

interactive(children=(FloatSlider(value=0.0, description='h (fs):', layout=Layout(width='500px'), max=50.0, mi…

# Autocorrelación

In [ ]:
npts = 500
t = np.linspace(-50, 50, npts)
dt = t[1] - t[0]               # time step (fs) — used by representacion
ix = np.arange(npts)
wf=2*np.pi/2
f = np.exp(-t**2 / 50)*np.exp(1j*wf*t)
g=f

full_corr = full_correlation(f, g)
ih_axis = np.arange(-npts // 2, npts // 2)

slider = widgets.FloatSlider(
    value=0.0, min=t[0], max=t[-1], step=dt,
    description='h (fs):', continuous_update=True,
    layout=widgets.Layout(width='500px')
)

widgets.interact(representacion, ih_time=slider)
plt.close()

interactive(children=(FloatSlider(value=0.0, description='h (fs):', layout=Layout(width='500px'), max=50.0, mi…